# Experiment 6 — Coordination Mechanisms: Centralized and Decentralized Task Allocation
**Aim:** To allocate a set of tasks among a set of agents under two coordination mechanisms — a central coordinator holding global state, and agents negotiating locally by announcing intentions — and to compare the two by solution quality, number of messages exchanged and behaviour when an agent fails.

## Procedure and Result

### Step 1 — Place agents and tasks on a grid and build the cost matrix
The cost of an agent performing a task is the Manhattan distance between them.

In [1]:
import itertools, math, random, time
import pandas as pd

def manhattan(p, q):
    return abs(p[0] - q[0]) + abs(p[1] - q[1])

def build_cost_matrix(agent_pos, task_pos):
    return [[manhattan(a, t) for t in task_pos] for a in agent_pos]

# Fixed positions (x, y) on the grid
agent_pos = [(0, 0), (3, 0), (0, 5), (7, 5)]      # A0..A3
task_pos  = [(2, 0), (6, 0), (1, 6), (8, 3)]      # T0..T3

agent_ids = list(range(len(agent_pos)))
C = build_cost_matrix(agent_pos, task_pos)

print("Agent positions:", {f"A{i}": p for i, p in enumerate(agent_pos)})
print("Task positions :", {f"T{j}": p for j, p in enumerate(task_pos)})
print("\nCost matrix (Manhattan distance):")
cost_df = pd.DataFrame(C, index=[f"A{i}" for i in agent_ids], columns=[f"T{j}" for j in range(len(task_pos))])
display(cost_df)

Agent positions: {'A0': (0, 0), 'A1': (3, 0), 'A2': (0, 5), 'A3': (7, 5)}
Task positions : {'T0': (2, 0), 'T1': (6, 0), 'T2': (1, 6), 'T3': (8, 3)}

Cost matrix (Manhattan distance):


,T0,T1,T2,T3
A0,2,6,7,11
A1,1,3,8,8
A2,7,11,2,10
A3,10,6,7,3


### Step 2 — Algorithm 1: Centralized allocation
Every agent sends one status report to the coordinator; the coordinator enumerates all assignments of distinct tasks to agents, keeps the cheapest, and sends one instruction back to each agent (messages = 2 per agent).

In [2]:
def centralized(C, ids):
    """C[i][j] = cost of agent ids[i] doing task j. Exhaustive search by the coordinator."""
    n, m = len(ids), len(C[0])
    best_cost, best_assign, evaluated = math.inf, None, 0
    if n <= m:
        # choose a distinct task for every agent
        for perm in itertools.permutations(range(m), n):
            evaluated += 1
            cost = sum(C[i][perm[i]] for i in range(n))
            if cost < best_cost:
                best_cost, best_assign = cost, {ids[i]: perm[i] for i in range(n)}
    else:
        # more agents than tasks: choose a distinct agent for every task
        for perm in itertools.permutations(range(n), m):
            evaluated += 1
            cost = sum(C[perm[j]][j] for j in range(m))
            if cost < best_cost:
                best_cost, best_assign = cost, {ids[perm[j]]: j for j in range(m)}
    messages = 2 * n      # one status report from each agent + one instruction back to it
    return {"assignment": best_assign, "cost": best_cost, "evaluated": evaluated, "messages": messages}

def show_assignment(assign):
    return ", ".join(f"A{a}->T{t}" for a, t in sorted(assign.items()))

cen = centralized(C, agent_ids)
print("CENTRALIZED ALLOCATION")
print("Assignment           :", show_assignment(cen["assignment"]))
print("Total cost           :", cen["cost"])
print("Assignments evaluated:", cen["evaluated"])
print("Messages             :", cen["messages"])

CENTRALIZED ALLOCATION
Assignment           : A0->T0, A1->T1, A2->T2, A3->T3
Total cost           : 10
Assignments evaluated: 24
Messages             : 8


### Step 3 — Algorithm 2: Decentralized allocation by announced intentions
In each round every unassigned agent picks the cheapest free task (ties by task id) and broadcasts it to the other agents (one message per recipient). Each contested task goes to the lowest announced cost (ties by agent id); winners are committed and the losers repeat with the remaining tasks.

In [3]:
def decentralized(C, ids):
    n, m = len(ids), len(C[0])
    free_tasks = set(range(m))
    unassigned = list(range(n))            # row indices of agents still without a task
    assignment, total, messages = {}, 0, 0
    rounds = []
    r = 0
    while unassigned and free_tasks:
        r += 1
        # 1. every unassigned agent announces its intention
        bids = {}
        for i in unassigned:
            t = min(free_tasks, key=lambda t: (C[i][t], t))
            bids[i] = (t, C[i][t])
            messages += n - 1              # broadcast to every other agent
        # 2. conflict rule: lowest cost wins, ties broken by agent identifier
        winners = {}
        for i, (t, c) in bids.items():
            if t not in winners or (c, ids[i]) < (winners[t][1], ids[winners[t][0]]):
                winners[t] = (i, c)
        for t, (i, c) in winners.items():
            assignment[ids[i]] = t
            total += c
            free_tasks.discard(t)
            unassigned.remove(i)
        rounds.append({
            "round": r,
            "bids": [(f"A{ids[i]}", f"T{t}", c) for i, (t, c) in bids.items()],
            "committed": [(f"A{ids[i]}", f"T{t}", c) for t, (i, c) in winners.items()],
        })
    return {"assignment": assignment, "cost": total, "messages": messages,
            "rounds": rounds, "num_rounds": r, "unserved_tasks": sorted(free_tasks)}

dec = decentralized(C, agent_ids)
print("DECENTRALIZED ALLOCATION")
for rd in dec["rounds"]:
    print(f"\nRound {rd['round']}")
    for a, t, c in rd["bids"]:
        print(f"   bid      : {a} announces {t} at cost {c}")
    for a, t, c in rd["committed"]:
        print(f"   committed: {a} keeps {t} (cost {c})")
print("\nAssignment :", show_assignment(dec["assignment"]))
print("Total cost :", dec["cost"])
print("Messages   :", dec["messages"])
print("Rounds     :", dec["num_rounds"])

DECENTRALIZED ALLOCATION

Round 1
   bid      : A0 announces T0 at cost 2
   bid      : A1 announces T0 at cost 1
   bid      : A2 announces T2 at cost 2
   bid      : A3 announces T3 at cost 3
   committed: A1 keeps T0 (cost 1)
   committed: A2 keeps T2 (cost 2)
   committed: A3 keeps T3 (cost 3)

Round 2
   bid      : A0 announces T1 at cost 6
   committed: A0 keeps T1 (cost 6)

Assignment : A0->T1, A1->T0, A2->T2, A3->T3
Total cost : 12
Messages   : 15
Rounds     : 2


### Step 4 — Side-by-side comparison

In [4]:
cmp_df = pd.DataFrame({
    "Centralized": [show_assignment(cen["assignment"]), cen["cost"], cen["messages"], "-", cen["evaluated"]],
    "Decentralized": [show_assignment(dec["assignment"]), dec["cost"], dec["messages"], dec["num_rounds"], "-"],
}, index=["Assignment", "Total cost", "Messages", "Rounds", "Assignments evaluated"])
display(cmp_df)

print("\nPer-agent cost (centralized vs decentralized):")
rows = []
for a in agent_ids:
    cc, dc = C[a][cen["assignment"][a]], C[a][dec["assignment"][a]]
    rows.append([f"A{a}", f"T{cen['assignment'][a]}", cc, f"T{dec['assignment'][a]}", dc, dc - cc])
display(pd.DataFrame(rows, columns=["Agent", "Central task", "Central cost", "Decentral task", "Decentral cost", "Difference"]))

,Centralized,Decentralized
Assignment,"A0->T0, A1->T1, A2->T2, A3->T3","A0->T1, A1->T0, A2->T2, A3->T3"
Total cost,10,12
Messages,8,15
Rounds,-,2
Assignments evaluated,24,-



Per-agent cost (centralized vs decentralized):


,Agent,Central task,Central cost,Decentral task,Decentral cost,Difference
0,A0,T0,2,T1,6,4
1,A1,T1,3,T0,1,-2
2,A2,T2,2,T2,2,0
3,A3,T3,3,T3,3,0


## Observations

**Difference in total cost.** The centralized coordinator returned the optimal total cost of 10 (A0->T0, A1->T1, A2->T2, A3->T3), while the decentralized protocol ended at 12 (A0->T1, A1->T0, A2->T2, A3->T3), i.e. 20% above the optimum. The extra cost is paid by **A0**: it performs T1 at cost 6 instead of T0 at cost 2 (+4), while A1 is actually better off (cost 1 instead of 3, i.e. -2). A2 and A3 are unaffected; the net loss is +2.

**The round that committed to an expensive choice.** It is **Round 1**. A0 and A1 both announced T0 (costs 2 and 1). The conflict rule gave T0 to A1 because 1 < 2, and A1, A2 and A3 were committed. That was locally correct but globally wrong: T0 was the only cheap task for A0, and A1 had a cheap alternative (T1 at cost 3) that it never considered because only the cheapest task is announced. In Round 2 only T1 was left, so A0 had to take it at cost 6.

**Growth of the message count.** The centralized scheme uses 2 messages per agent (one status report and one instruction), so it grows linearly: 2n (8 for four agents; 6, 10, 12, 14, 16 for n = 3, 5, 6, 7, 8 in Exercise 1). In the decentralized scheme every announcement is broadcast to the other n-1 agents and every round repeats this for the agents still unassigned, so the count is roughly n(n-1) in the first round plus further rounds (12 + 3 = 15 here). It therefore grows roughly quadratically with n (about 88 messages at n = 8 against 16 for the centralized scheme). The price of having no coordinator is paid in messages.

**When an agent stops responding.** In the centralized scheme the coordinator does not receive that agent's report, so it has an incomplete cost matrix; it must wait for a time-out and then re-run the search without that agent, and any instruction already sent to it is lost (its task stays unserved until re-allocation). In the decentralized scheme the other agents simply carry on: the silent agent never announces anything, so it never wins a task, and the remaining agents allocate among themselves in the same rounds. No agent is indispensable, so the protocol keeps working, whereas the centralized scheme depends on a single decision point. Exercise 2 shows the result of removing an agent before allocation.

## Exercises

### Exercise 1 — Random instances, three to eight agents and tasks
For each size n = 3..8, 20 random instances are generated (agents and tasks at random cells of a 10 × 10 grid); averages are reported. The exhaustive search time is measured so that the practical limit can be stated.

In [5]:
random.seed(401)
GRID, TRIALS = 10, 20

def random_instance(n):
    agents = [(random.randrange(GRID), random.randrange(GRID)) for _ in range(n)]
    tasks  = [(random.randrange(GRID), random.randrange(GRID)) for _ in range(n)]
    return build_cost_matrix(agents, tasks)

rows = []
for n in range(3, 9):
    opt = dc = cm = dm = ev = tm = 0
    for _ in range(TRIALS):
        Cn = random_instance(n)
        ids = list(range(n))
        t0 = time.perf_counter()
        c = centralized(Cn, ids)
        tm += time.perf_counter() - t0
        d = decentralized(Cn, ids)
        opt += c["cost"]; dc += d["cost"]
        cm += c["messages"]; dm += d["messages"]; ev += c["evaluated"]
    rows.append({
        "n (agents = tasks)": n,
        "Optimal cost": opt / TRIALS,
        "Decentralized cost": dc / TRIALS,
        "Excess over optimum (%)": 100 * (dc - opt) / opt,
        "Messages (central)": cm / TRIALS,
        "Messages (decentral)": dm / TRIALS,
        "Assignments evaluated": ev / TRIALS,
        "Search time (s)": tm / TRIALS,
    })
ex1 = pd.DataFrame(rows).set_index("n (agents = tasks)")
display(ex1.round({"Optimal cost": 2, "Decentralized cost": 2, "Excess over optimum (%)": 2, "Messages (central)": 2, "Messages (decentral)": 2, "Assignments evaluated": 0, "Search time (s)": 4}))

,Optimal cost,Decentralized cost,Excess over optimum (%),Messages (central),Messages (decentral),Assignments evaluated,Search time (s)
n (agents = tasks),,,,,,,
3,13.20,13.90,5.30,6.0,8.00,6.0,0.0000
4,16.80,17.50,4.17,8.0,17.55,24.0,0.0001
5,21.00,21.80,3.81,10.0,29.80,120.0,0.0001
6,26.15,28.85,10.33,12.0,54.25,720.0,0.0006
7,25.25,27.65,9.50,14.0,69.30,5040.0,0.0035
8,25.40,28.00,10.24,16.0,87.85,40320.0,0.0266


In [6]:
# Growth of the exhaustive search: n! assignments, extrapolated from the measured time per assignment at n = 8
per_assignment = ex1.loc[8, "Search time (s)"] / ex1.loc[8, "Assignments evaluated"]
proj = []
for n in range(8, 16):
    secs = math.factorial(n) * per_assignment
    proj.append({"n": n, "Assignments (n!)": math.factorial(n), "Estimated search time (s)": secs,
                 "Estimated search time (min)": secs / 60})
display(pd.DataFrame(proj).set_index("n").round(2))

,Assignments (n!),Estimated search time (s),Estimated search time (min)
n,,,
8,40320,0.03,0.00
9,362880,0.24,0.00
10,3628800,2.39,0.04
11,39916800,26.29,0.44
12,479001600,315.45,5.26
13,6227020800,4100.90,68.35
14,87178291200,57412.62,956.88
15,1307674368000,861189.28,14353.15


**Result of Exercise 1**

**Result of Exercise 1.** (20 random instances per size, 10 x 10 grid, seed 401; the tables above give the numbers.)

- **Optimal vs decentralized cost:** the decentralized allocation was always somewhat more expensive than the optimum, by about 4% to 10% depending on size: 5.3% at n = 3, 4.2% at n = 4, 3.8% at n = 5, 10.3% at n = 6, 9.5% at n = 7 and 10.2% at n = 8. The gap is larger for the bigger instances (about 10% at n = 6 to 8) because there are more chances for an early greedy commitment to take a task that another agent needed.
- **Messages:** centralized messages are exactly 2n (6 to 16). Decentralized messages grow much faster: 8.0, 17.6, 29.8, 54.3, 69.3 and 87.9 for n = 3 to 8 (roughly quadratic, since each round is a broadcast by every unassigned agent).
- **Assignments evaluated by the coordinator:** n! = 6, 24, 120, 720, 5 040 and 40 320 for n = 3 to 8. This is factorial growth, while the message count grows only linearly.
- **Size at which exhaustive search stops being practical:** the search at n = 8 takes about 0.03 s in this implementation. Extrapolating the measured time per assignment, n = 10 needs about 2 s, n = 11 about 25 s, n = 12 about 5 minutes, n = 13 more than an hour and n = 14 about 15 hours. Exhaustive centralized search therefore stops being practical at about **n = 12** (with n = 10 to 11 being the last comfortable sizes), and it is hopeless beyond n = 13 without a smarter algorithm.
- **Percentage by which the decentralized allocation exceeds the optimum:** about 4% to 10% on average (about 10% for n = 6 to 8, and roughly 7% averaged over all six sizes).

### Exercise 2 — Remove one agent before allocation, and coordinator failure
Agent A1 is removed from the main instance before allocation starts; both mechanisms are re-run on the remaining agents (original identifiers kept).

In [7]:
REMOVED = 1
ids2 = [a for a in agent_ids if a != REMOVED]
C2 = [C[a] for a in ids2]
m = len(task_pos)

cen2 = centralized(C2, ids2)
dec2 = decentralized(C2, ids2)

print(f"Agent A{REMOVED} removed. Remaining agents:", [f"A{a}" for a in ids2])
display(pd.DataFrame(C2, index=[f"A{a}" for a in ids2], columns=[f"T{j}" for j in range(m)]))

for name, res in (("CENTRALIZED", cen2), ("DECENTRALIZED", dec2)):
    unserved = sorted(set(range(m)) - set(res["assignment"].values()))
    print(f"\n{name}")
    print("Assignment    :", show_assignment(res["assignment"]))
    print("Total cost    :", res["cost"])
    print("Messages      :", res["messages"])
    print("Unserved tasks:", [f"T{t}" for t in unserved])

print("\nFor reference, with all four agents: central cost =", cen["cost"], ", decentral cost =", dec["cost"])

Agent A1 removed. Remaining agents: ['A0', 'A2', 'A3']


,T0,T1,T2,T3
A0,2,6,7,11
A2,7,11,2,10
A3,10,6,7,3



CENTRALIZED
Assignment    : A0->T0, A2->T2, A3->T3
Total cost    : 7
Messages      : 6
Unserved tasks: ['T1']

DECENTRALIZED
Assignment    : A0->T0, A2->T2, A3->T3
Total cost    : 7
Messages      : 6
Unserved tasks: ['T1']

For reference, with all four agents: central cost = 10 , decentral cost = 12


**Result of Exercise 2**

**Result of Exercise 2.** Agent A1 was removed before allocation, leaving A0, A2 and A3 for four tasks.

- **Centralized:** A0->T0, A2->T2, A3->T3, total cost 7, 6 messages (2 per remaining agent), 24 assignments evaluated. **T1 is left unserved.**
- **Decentralized:** A0->T0, A2->T2, A3->T3, total cost 7, 6 messages (three announcements of two recipients each, in a single round). **T1 is left unserved.**
- Both mechanisms give the same assignment and cost here. Without A1 nobody competes with A0 for T0, so the greedy choices happen to be optimal and no conflict occurs. Since there are only three agents for four tasks, one task must stay unserved; T1 is the one left because it is the most expensive choice for every remaining agent (6, 11 and 6) once the cheaper tasks are taken. Compared with the full instance (10 and 12), the total cost drops from 10 to 7 and from 12 to 7 only because one task is no longer performed.

**If the coordinator itself fails.**

- *Centralized:* the coordinator is a single point of failure. The agents' reports go nowhere and no instruction is ever sent, so no task is allocated at all and every agent stays idle.
- *Decentralized:* there is no coordinator, so nothing changes. The agents keep announcing intentions to each other and resolve conflicts with the fixed rule, so all tasks are still allocated.
- *Change to the centralized scheme to survive this:* replicate the coordinator. Keep one or more standby coordinators that receive the same status reports as the primary (every agent sends its report to all of them), and have them monitor the primary with heartbeats or time-outs. If the primary goes silent, the standby with the lowest identifier (chosen by a simple leader election) takes over, runs the same search on the cost matrix it already holds and sends the instructions. This removes the single point of failure at the price of extra messages (one report per replica) and the need to keep the replicas consistent.